In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_RAW = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")

attrition_hr = pd.read_csv(
    DATA_RAW / "WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

print("Shape:", attrition_hr.shape)
print("Target:", attrition_hr["Attrition"].value_counts())

Shape: (1470, 35)
Target: Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [3]:
drop_cols = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

attrition_model = attrition_hr.drop(columns=drop_cols)

print("Shape after removing irrelevant columns:", attrition_model.shape)

Shape after removing irrelevant columns: (1470, 31)


In [4]:
attrition_model["Attrition"] = (
    attrition_model["Attrition"]
    .map({"No": 0, "Yes": 1})
)

print(attrition_model["Attrition"].value_counts())

Attrition
0    1233
1     237
Name: count, dtype: int64


In [5]:
X = attrition_model.drop(columns=["Attrition"])
y = attrition_model["Attrition"]

print("X:", X.shape)
print("y:", y.shape)

X: (1470, 30)
y: (1470,)


In [6]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

Categorical: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Numerical: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training: (1176, 30)
Testing : (294, 30)

Train target distribution:
Attrition
0    0.838435
1    0.161565
Name: proportion, dtype: float64

Test target distribution:
Attrition
0    0.840136
1    0.159864
Name: proportion, dtype: float64


In [8]:
X_train.to_csv(DATA_PROCESSED / "X_train.csv", index=False)
X_test.to_csv(DATA_PROCESSED / "X_test.csv", index=False)

y_train.to_csv(DATA_PROCESSED / "y_train.csv", index=False)
y_test.to_csv(DATA_PROCESSED / "y_test.csv", index=False)

print("Prepared datasets saved.")

Prepared datasets saved.


In [9]:
print("=" * 80)
print("ATTRITION PREPARATION COMPLETE")
print("=" * 80)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nMissing values:")
print(X_train.isna().sum().sum(), "in training")
print(X_test.isna().sum().sum(), "in testing")

print("\nTarget distribution:")
print(y_train.value_counts())

ATTRITION PREPARATION COMPLETE
X_train: (1176, 30)
X_test : (294, 30)
y_train: (1176,)
y_test : (294,)

Missing values:
0 in training
0 in testing

Target distribution:
Attrition
0    986
1    190
Name: count, dtype: int64


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X = attrition_hr.drop("Attrition", axis=1)
y = attrition_hr["Attrition"]

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = X.select_dtypes(exclude=["object"]).columns.tolist()

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_cat = encoder.fit_transform(X[categorical_features])

X_cat = pd.DataFrame(
    X_cat,
    columns=encoder.get_feature_names_out(categorical_features),
    index=X.index
)

X_final = pd.concat(
    [X[numerical_features], X_cat],
    axis=1
)

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

X_train: (1176, 55)
X_test : (294, 55)
Categorical features: 8
Numerical features: 26


In [11]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(encoder, MODEL_DIR / "attrition_encoder.pkl")

print("Encoder saved successfully.")

Encoder saved successfully.


In [11]:
import joblib

DATA_PROCESSED = "../data/processed"

X_train.to_csv(f"{DATA_PROCESSED}/X_train.csv", index=False)
X_test.to_csv(f"{DATA_PROCESSED}/X_test.csv", index=False)
y_train.to_csv(f"{DATA_PROCESSED}/y_train.csv", index=False)
y_test.to_csv(f"{DATA_PROCESSED}/y_test.csv", index=False)

joblib.dump(encoder, f"{DATA_PROCESSED}/attrition_encoder.pkl")

print("Attrition preparation files saved successfully.")

Attrition preparation files saved successfully.


In [12]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Test :", X_test.isna().sum().sum())

print("\nTarget distribution:")
print(y_train.value_counts())

X_train: (1176, 55)
X_test : (294, 55)
y_train: (1176,)
y_test : (294,)

Missing values:
Train: 0
Test : 0

Target distribution:
Attrition
No     986
Yes    190
Name: count, dtype: int64
